# SERS-D2DNet translation training

**SERS-D2DNet + SuperRaman · v0.2.0**

[Notebook guide](../docs/NOTEBOOK_GUIDE.md) · [Paper alignment](../docs/PAPER_IMPLEMENTATION_MAP.md) · [README](../README.md)

## Goal
Control source-to-Renishaw fitting for every selected portable device, with actual fit counts and checkpoint selection visible.

## Setup
Select the Python environment where `.[notebooks]` is installed. Edit `CONFIG` below to select the demo or your research configuration. Every notebook opens the same persistent run. Configuration changes require a new `run_directory`. The default executes a synthetic, one-epoch, width-2 demonstration across all four portable devices.

In [1]:
CONFIG = "configs/workflow_demo.json"
RUN_RESEARCH_TRAINING = False

In [2]:
import numpy as np
from sersd2d_superraman import __version__
from sersd2d_superraman.provenance import read_json
from sersd2d_superraman.data import load_batch
from sersd2d_superraman.notebook import (
    open_workflow,
    display_table,
    setup_plotting,
    training_plot,
    spectrum_plot,
    confusion_plot,
)

workflow = open_workflow(CONFIG)
PROJECT, settings = workflow.project, workflow.config
MODE = settings["mode"]
RUN_TRAINING = MODE == "demo" or RUN_RESEARCH_TRAINING
setup_plotting()
display_table(
    ["Control", "Value"],
    [
        ["Software", __version__],
        ["Mode", MODE],
        ["Devices", ", ".join(settings["sources"])],
        [
            "Translation width / epochs",
            f"{settings['translation_config']['model']['width']} / {settings['translation_config']['training']['epochs']}",
        ],
        ["Classifier epochs", settings["classifier_config"]["training"]["epochs"]],
        [
            "Classifier / translation device",
            f"{settings['classifier_device']} / {settings['translation_device']}",
        ],
        ["Research training requested", RUN_RESEARCH_TRAINING],
    ],
)

| Control | Value |
| --- | --- |
| Software | 0.2.0 |
| Mode | demo |
| Devices | Enwave, Tec5, Fdefender, RID |
| Translation width / epochs | 2 / 1 |
| Classifier epochs | 1 |
| Classifier / translation device | cpu / cpu |
| Research training requested | False |

### Read the execution scope
The demo verifies software behavior. Its plots and scores are synthetic and have no scientific performance interpretation. Research mode uses authorized preprocessed arrays and the recovered width-128 graph. Published-result reproduction remains unresolved; review the paper alignment document before reporting results.

## Steps
### 1. Prepare and audit the input roles
The interface accepts preprocessed `[N,1400]` spectra in `[0,1]`. Rows retain device, analyte, sample identity, optional Raman axis, and optional acquisition group. Class-matched training pairs are physically unpaired. Split checks keep duplicates and acquisition groups together.

In [3]:
counts = workflow.prepare()
display_table(
    ["Device", "Fit", "Validation", "Test"],
    [[d, r["train"], r["validation"], r["test"]] for d, r in counts.items()],
)

synthetic: verified existing artifacts


prepare: verified existing artifacts


| Device | Fit | Validation | Test |
| --- | --- | --- | --- |
| Renishaw | 60 | 20 | 20 |
| Enwave | 60 | 20 | 20 |
| Tec5 | 60 | 20 | 20 |
| Fdefender | 60 | 20 | 20 |
| RID | 60 | 20 | 20 |

### 2. Train one translator for each portable device
Every translator learns a source-to-Renishaw mapping independently. Deep supervision uses named heads; inference selects only the 1400-bin `out` head. The recovered graph is retained. A maximum of 50 class-matched fit pairs is requested in research mode; actual per-class counts are recorded, since small independent fit splits can contain fewer.

In [4]:
translation_runs = workflow.train_translations() if RUN_TRAINING else {}
if not RUN_TRAINING:
    print("Research translation training is disabled. Set RUN_RESEARCH_TRAINING = True in Setup to run it.")
display_table(
    ["Device", "Best epoch", "Validation MAE", "Fit pairs/class (min–max)"],
    [
        [
            d,
            r["best_epoch"],
            round(r["best_validation_out_mae"], 5),
            str(min(r["fit_class_counts"].values())) + "–" + str(max(r["fit_class_counts"].values())),
        ]
        for d, r in translation_runs.items()
    ],
)

synthetic: verified existing artifacts


prepare: verified existing artifacts


translation:Enwave: verified existing artifacts


translation:Tec5: verified existing artifacts


translation:Fdefender: verified existing artifacts


translation:RID: verified existing artifacts


| Device | Best epoch | Validation MAE | Fit pairs/class (min–max) |
| --- | --- | --- | --- |
| Enwave | 1 | 0.15752 | 1–1 |
| Tec5 | 1 | 0.15728 | 1–1 |
| Fdefender | 1 | 0.15733 | 1–1 |
| RID | 1 | 0.15733 | 1–1 |

### Check saved full-resolution output contracts

In [5]:
if translation_runs:
    print(
        {
            d: {
                "parameters": read_json(workflow.output / "translation" / d / "checkpoint/metadata.json")[
                    "parameters"
                ],
                "outputs": read_json(workflow.output / "translation" / d / "checkpoint/metadata.json")[
                    "output_names"
                ],
            }
            for d in settings["sources"]
        }
    )

{'Enwave': {'parameters': 15876, 'outputs': ['level1', 'level2', 'level3', 'out']}, 'Tec5': {'parameters': 15876, 'outputs': ['level1', 'level2', 'level3', 'out']}, 'Fdefender': {'parameters': 15876, 'outputs': ['level1', 'level2', 'level3', 'out']}, 'RID': {'parameters': 15876, 'outputs': ['level1', 'level2', 'level3', 'out']}}


## Checks
Artifact reuse checks file hashes and input/configuration identities. Test rows are evaluated after checkpoint selection; they do not choose the models.

In [6]:
display_table(["Stage", "Status"], workflow.status().items())

| Stage | Status |
| --- | --- |
| bundle | completed |
| classifier | completed |
| metrics:Enwave | completed |
| metrics:Fdefender | completed |
| metrics:RID | completed |
| metrics:Tec5 | completed |
| predict:Enwave | completed |
| predict:Fdefender | completed |
| predict:RID | completed |
| predict:Tec5 | completed |
| prepare | completed |
| synthetic | completed |
| translation:Enwave | completed |
| translation:Fdefender | completed |
| translation:RID | completed |
| translation:Tec5 | completed |

## Next Steps
For real experiments, copy `configs/workflow_research.json` to `configs/local/my_run.json`, set the data path and a new run directory, and select it in `CONFIG`. Set RUN_RESEARCH_TRAINING = True explicitly after reviewing the displayed settings. See the [notebook guide](../docs/NOTEBOOK_GUIDE.md) for stage navigation and interpretation.